# Lab 05-09 · Requirements, tests, coverage and defect filters

Companion to chapter [05-09](../../chapters/05-09-software-engineering-practices.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction first. Everything is deterministic (fixed seeds).

In [1]:
import sys
import random
from itertools import combinations, product

## Part 1 — Requirements to tests, and a planted bug

Requirement (from Example 05-09.1): assert LOW_BATT below 3.30 V; deassert at or above 3.40 V; hold in between. Three implementations are tested: the correct one and two with plausible slips.

**🔮 Predict:** which of the eight boundary tests fail for each buggy version?

In [2]:
def correct(v, state):
    if v < 3.30: return True
    if v >= 3.40: return False
    return state

def bug_le(v, state):                 # <= instead of < at the lower threshold
    if v <= 3.30: return True
    if v >= 3.40: return False
    return state

def bug_no_hold(v, state):            # forgets the hysteresis: clears as soon as v >= 3.30
    return v < 3.30

cases = [(3.29, False, True), (3.30, False, False), (3.31, False, False),
         (3.39, True, True), (3.40, True, False), (3.41, True, False),
         (3.35, True, True), (3.35, False, False)]
for f in (correct, bug_le, bug_no_hold):
    failed = [(v, s) for v, s, want in cases if f(v, s) != want]
    print(f"{f.__name__:12} {len(failed)} failed: {failed}")

correct      0 failed: []
bug_le       1 failed: [(3.3, False)]
bug_no_hold  2 failed: [(3.39, True), (3.35, True)]


`bug_le` fails exactly one test, the one *at* 3.30 V: a test value one step away would never have noticed. `bug_no_hold` fails the two tests that check the hold behavior with the alarm already on (3.39 V and 3.35 V with state True), which is precisely what the hysteresis requirement adds. Each test exists to catch a specific class of mistake.

## Part 2 — Measuring line coverage yourself

Coverage tools work by asking the interpreter to report every line it executes. Python exposes that hook as `sys.settrace`.

**🔮 Predict:** which lines of `debounce` does the first test set miss?

In [3]:
def debounce(samples, needed=3):
    count, state, out = 0, False, []
    for v in samples:
        if v < 3.30:
            count += 1
            if count >= needed:
                state = True
        else:
            count = 0
            if v >= 3.40:
                state = False
        out.append(state)
    return out

def coverage(func, calls):
    hit = set()
    def tracer(frame, event, arg):
        if frame.f_code is func.__code__ and event == "line":
            hit.add(frame.f_lineno)
        return tracer
    sys.settrace(tracer)
    try:
        for args in calls:
            func(*args)
    finally:
        sys.settrace(None)
    first = func.__code__.co_firstlineno
    body = {line for _, _, line in func.__code__.co_lines() if line and line > first}   # executable lines only
    return sorted(l - first for l in body - hit), len(hit & body) / len(body)

missed, frac = coverage(debounce, [([3.25, 3.25],), ([3.35, 3.36],)])
print("first set: missed body lines", missed, f"coverage {frac:.0%}")
missed, frac = coverage(debounce, [([3.25, 3.25, 3.25, 3.41],)])
print("one better test: missed", missed, f"coverage {frac:.0%}")

first set: missed body lines [6, 10] coverage 82%
one better test: missed [] coverage 100%


The first set never reaches three consecutive low samples, so `state = True` (line 6 of the body) never runs, and it never sees a sample at or above 3.40 V, so `state = False` (line 10) never runs either: 9 of the 11 executable lines, 82%. (The `else:` line is not executable, so the tracer, like real coverage tools, does not count it.) A single, well-chosen sequence (three low samples, then a high one) reaches every line. Coverage pointed at the gaps; it did not say whether the outputs were *checked*, which is the tests' job.

## Part 3 — MC/DC for a four-condition interlock

`run = power_ok and door_closed and (temp_ok or override)`

**🔮 Predict:** the minimum number of MC/DC tests, and the number of exhaustive tests.

In [4]:
def run(p, d, t, o):
    return p and d and (t or o)

def mcdc_ok(tests, f, n):
    return all(any(all(a[i] == b[i] for i in range(n) if i != k) and a[k] != b[k] and f(*a) != f(*b)
                   for a, b in combinations(tests, 2)) for k in range(n))

cases = list(product([0, 1], repeat=4))
for size in range(1, 17):
    sets = [s for s in combinations(cases, size) if mcdc_ok(s, run, 4)]
    if sets:
        print(f"minimum MC/DC: {size} tests (of {len(cases)} combinations); {len(sets)} minimal sets, e.g. {sets[0]}")
        break

minimum MC/DC: 5 tests (of 16 combinations); 4 minimal sets, e.g. ((0, 1, 0, 1), (1, 0, 0, 1), (1, 1, 0, 0), (1, 1, 0, 1), (1, 1, 1, 0))


Five tests ($N + 1$) suffice, out of 16 combinations. Several different minimal sets exist; choosing among them is where engineering judgment (which combinations are physically meaningful, which are hard to set up on the rig) comes in.

## Part 4 — Which defect filter matters most?

**🔮 Predict:** removing which single filter increases escapes the most?

In [5]:
filters = {"review": 0.6, "static analysis": 0.3, "unit tests": 0.5, "integration": 0.4, "system": 0.3}
def escapes(active, defects=100):
    f = 1.0
    for name in active:
        f *= 1 - filters[name]
    return defects * f

base = escapes(filters)
print(f"all filters: {base:.2f} of 100 defects escape")
for name in filters:
    print(f"without {name:15}: {escapes([n for n in filters if n != name]):5.2f} (x{escapes([n for n in filters if n != name]) / base:.2f})")

all filters: 5.88 of 100 defects escape
without review         : 14.70 (x2.50)
without static analysis:  8.40 (x1.43)
without unit tests     : 11.76 (x2.00)
without integration    :  9.80 (x1.67)
without system         :  8.40 (x1.43)


Removing the strongest filter (review, 60%) multiplies escapes by $1/(1 - 0.6) = 2.5$; removing a 30% filter multiplies them by only 1.43. Under independence, a filter's value is set by its own catch rate alone; in reality, a filter that is *different in kind* from the others (static analysis, say) is worth more than its nominal rate suggests, because its blind spots are different (Example 05-09.4).

## Your turn

1. Add a third buggy version of the battery monitor that uses `>` instead of `>=` at 3.40 V. Which test catches it?
2. In Part 3, change the interlock to `power_ok and (door_closed or maintenance) and (temp_ok or override)` and find the MC/DC minimum.